# M4 — Validation: what the prototype may claim

**retrieve-or-bust · branch `rt-elastic-prototype` · 2026-08-06**

M4 runs the design §6 protocol and decides what the Week-1 prototype is allowed to
say. Every model on identical data, scored in `rrs`, broken out per wavelength, per
solar zenith and per `B_p` bin, on both held-out splits, plus speed and the gradient
gate.

The milestone's real work was not measuring — it was **choosing the comparison**. M3
reported the hybrid beating standard Gordon by 24×. M4 adds **O25** (Pitarch et al.
2025), and against that the margin is **2.3×**. Same model, same data, different
benchmark; the second number is the honest one.

And on the split where the prototype is most exposed — an unseen solar zenith — the
hybrid **loses** to O25, which is also the only model there whose answer does not
depend on a random seed.

| what M4 added | where |
|---|---|
| O25, refit on L23 | `robust/rt/baselines.py` |
| the §6 protocol: per-λ / zenith / `B_p`, throughput, gradients | `robust/rt/validation.py` |
| the out-of-domain fallback (JXP's Q7) | `robust/rt/hybrid.py` |
| the runner and its committed output | `design/py/run_validation.py`, `design/validation/` |

Earlier notebooks cover the ground this one builds on:
[1](rt_elastic_coding_1.ipynb) (JAX, autodiff, float64),
[2](rt_elastic_coding_2.ipynb) (`rrs` vs `Rrs`, the splits, `B_p`'s narrow range),
[3](rt_elastic_coding_3.ipynb) (the ZTT backbone vs Gordon) and
[4](rt_elastic_coding_4.ipynb) (**the hybrid, specified stage by stage in its §1**).

In [ ]:
import sys
from pathlib import Path

REPO = next(
    p for p in [Path.cwd(), *Path.cwd().parents] if (p / "robust" / "__init__.py").exists()
)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

import jax
import jax.numpy as jnp
import numpy as np

from robust.rt import baselines as B
from robust.rt import conventions as C
from robust.rt import emulator as E
from robust.rt import hybrid as H
from robust.rt import validation as V
from robust.rt import ztt as Z
from robust.rt.data import l23 as L
from robust.rt.types import Geometry, IOPs, PhaseParams

print(f"jax {jax.__version__} on {jax.default_backend()}")

In [ ]:
import matplotlib as mpl
import matplotlib.pyplot as plt

# House style, unchanged from notebooks 1-4 so the five read as one set.
INK, INK_MUTED, GRID = "#1a1a1a", "#5c5c5c", "#dcdcdc"
# The CVD-validated categorical set (worst adjacent pair dE 21.9, protan). #56B4E9
# sits below 3:1 contrast on white, so every series is DIRECTLY LABELLED.
C_A, C_B, C_C = "#0072B2", "#D55E00", "#56B4E9"   # blue, vermillion, sky
MUTED = "#9a9a9a"   # not a category: de-emphasis, and seed replicates
mpl.rcParams.update({
    "figure.dpi": 110, "savefig.dpi": 110, "font.size": 10.5,
    "axes.edgecolor": INK_MUTED, "axes.labelcolor": INK, "text.color": INK,
    "xtick.color": INK_MUTED, "ytick.color": INK_MUTED,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.6,
    "legend.frameon": False, "figure.facecolor": "white", "axes.facecolor": "white",
})
print("style set")

## 1. The two models being compared, defined

This notebook spends its length comparing two things. Both have been built over
earlier milestones, so here they are stated plainly before any numbers arrive.

### O25 — Pitarch et al. (2025)

*Analytical modeling and correction of the ocean colour bidirectional reflectance
across water types*, **Remote Sens. Environ. 329, 114920**. It inherits its forward
form from Lee et al. (2011) and is calibrated on the synthetic multi-angular set
PB24 (Pitarch & Brando 2025). It is **the current state of the art in the elastic
Gordon → QAA lineage**: open source, integrated in NASA HyperCP and EUMETSAT ThoMaS,
and operational in OLCI Collection 4. That is why it, and not standard Gordon, is the
benchmark that decides whether this project's hybrid is worth anything.

The model is a **bivariate quadratic in the split backscatter albedos**:

$$R_{rs} = (G_{w0} + G_{w1}\,\omega_{bw})\,\omega_{bw}
         + (G_{p0} + G_{p1}\,\omega_{bp})\,\omega_{bp},
\qquad
\omega_{bw} = \frac{b_{bw}}{a + b_b},\quad
\omega_{bp} = \frac{b_{bp}}{a + b_b}$$

Every symbol: $a$ absorption, $b_{bw}$ the backscattering of pure water (a known
constant of the medium), $b_{bp}$ particulate backscattering, $b_b = b_{bw} + b_{bp}$.
So $\omega_{bw}$ and $\omega_{bp}$ are the water and particle shares of a
Gordon-style single-scattering albedo, kept apart rather than summed.

**Why that split is the whole idea.** Water and particles return light through
*different* volume scattering functions, so their contributions to $R_{rs}$ should not
share a coefficient. O25 gives each branch its own pair. And because this project
already carries `bb_w` and `bb_p` as separate fields (design §3), running O25 costs us
nothing — no $\gamma_b$ iteration, no approximation.

**The four $G$ coefficients depend on geometry and nothing else** — $(\theta_s,
\theta_v, \Delta\phi)$ — and are wavelength- and IOP-agnostic *by construction*.
Wavelength enters only through the IOPs. In the published model they are lookup tables
over 1300 geometries.

**What O25 cannot see**, and both are deliberate, not shortcomings of our
implementation:

- **No phase-function input at all.** Its coefficients were fitted on a set with
  *prescribed* Fournier-Forand phase functions, so phase-function shape is baked into
  the numbers rather than being adjustable. This is precisely the gap the ZTT backbone
  exists to address, and the reason O25 is a *benchmark* rather than a physical
  reference.
- **No wavelength dependence in the coefficients**, as above.

**Our O25 is a refit, and every table says so.** The published $G$ tables appear in
the paper only as plots, and are not in this repo — they live in the authors' code. So
`baselines.fit_o25` fits the four coefficients per solar zenith on L23's **training
split**, by closed-form weighted least squares, and everything is labelled **"O25
form, refit on L23"**. Two consequences worth carrying: it is not a statement about
the published model, and it has seen our training data, so its numbers are its *best
case*. It also has no $\theta_v$/$\Delta\phi$ axis, because L23 is nadir-only.

Finally, O25 is defined in **`Rrs`** (above water), not `rrs`, which is why
`Rrs_o25` is the primitive here and `rrs_o25` converts for scoring. Its stated validity
ceiling is `Rrs ≤ 0.06 sr⁻¹`; L23 reaches 0.0248, so nothing below extrapolates in
brightness.

### The hybrid — this project's model

$$R_{rs} = R_{rs}^{ZTT} + \Delta R_{rs},
\qquad rrs = rrs_{ZTT}\,(1 + \delta),
\qquad \delta = \delta_{\max}\tanh\bigl(\mathrm{MLP}(x)\bigr)$$

Two halves, and the division is the point:

- **An unfitted radiative-transfer backbone** — the Zaneveld–Twardowski–Tonizzo model
  transcribed in M2 ([notebook 3](rt_elastic_coding_3.ipynb)). Its coefficients come
  from the papers, not from L23, and it carries the **explicit backward volume
  scattering function** that O25 lacks: `B_p` is a real input, not an implicit
  assumption.
- **A small learned correction** — 417 parameters over seven dimensionless features,
  trained on the residual `rrs_L23 − rrs_ZTT` on the **training split only**, bounded
  by construction at $|\delta| < 0.5$, and zero at initialisation so an untrained
  hybrid *is* the backbone.

[Notebook 4](rt_elastic_coding_4.ipynb) **§1 specifies every stage** — the feature
map, the standardisation, the network, the bound, the assembly, the interface — and is
not repeated here.

The comparison this notebook runs is therefore not "network versus equation". It is
**12 fitted numbers with no phase-function input** against **417 fitted numbers on top
of unfitted physics that has one** — and the interesting questions are how much the
extra machinery buys, and where it stops being trustworthy.

| | O25 form, refit on L23 | the hybrid |
|---|---|---|
| fitted parameters | 12 (4 per solar zenith) | 417, plus the unfitted backbone |
| fitted on | L23 train split, weighted least squares | L23 train split, Adam on `rrms` |
| phase function `B_p` | **not an input** | an input to the backbone *and* the emulator |
| solar zenith | a coefficient row per angle | a `cos θ_s` feature |
| sensor zenith / azimuth | in the published model; **not in our refit** | carried, but constant in L23 |
| wavelength | only through the IOPs | a feature, and through the IOPs |
| defined in | `Rrs` | `rrs`, then through the interface |
| differentiable | yes, except at its table nodes (§6) | yes, ~1e-9 in every input (§6) |

In [ ]:
# What the two models are, on one water body -- concrete rather than abstract.
# Needs no reference data: a single plausible IOP set at three wavelengths.
demo_wave = jnp.asarray([440.0, 550.0, 660.0])
demo_iops = IOPs(
    a=jnp.asarray([0.045, 0.062, 0.400]),
    bb_w=C.bb_w(demo_wave),
    bb_p=jnp.asarray([0.0060, 0.0048, 0.0037]),
)
demo_phase = PhaseParams(B_p=jnp.asarray(0.0126))
demo_geom = Geometry.nadir(jnp.asarray(30.0))

print("O25's coefficients, as refit on L23 (baselines.O25_L23_REFIT):")
print(f"   {'theta_s':>8}{'Gw0':>10}{'Gw1':>10}{'Gp0':>10}{'Gp1':>10}")
for row in B.O25_L23_REFIT:
    print(f"   {row[0]:>8.0f}" + "".join(f"{v:>10.5f}" for v in row[1:]))
print("   -- geometry only: no wavelength, no B_p, and (in our refit) no view angle")

bb = demo_iops.bb
w_bw, w_bp = demo_iops.bb_w / (demo_iops.a + bb), demo_iops.bb_p / (demo_iops.a + bb)
print(f"\nthe two branch albedos at theta_s = 30 deg:")
print(f"   {'lambda':>8}{'omega_bw':>11}{'omega_bp':>11}{'O25 Rrs':>11}")
o25_demo = B.Rrs_o25(demo_iops, demo_phase, demo_geom, demo_wave)
for i, lam in enumerate(np.asarray(demo_wave)):
    print(f"   {lam:>8.0f}{float(w_bw[i]):>11.5f}{float(w_bp[i]):>11.5f}"
          f"{float(o25_demo[i]):>11.6f}")

packaged = E.load_default()
ztt_demo = Z.rrs_ZTT(demo_iops, demo_phase, demo_geom, demo_wave)
delta_demo = packaged.relative_delta(demo_iops, demo_phase, demo_geom, demo_wave)
print(f"\nthe hybrid, term by term (packaged weights, {sum(p.size for p in "
      f"jax.tree_util.tree_leaves(packaged.params))} parameters):")
print(f"   {'lambda':>8}{'rrs_ZTT':>11}{'delta':>10}{'rrs':>11}{'Rrs':>11}")
for i, lam in enumerate(np.asarray(demo_wave)):
    rrs = float(ztt_demo[i]) * (1 + float(delta_demo[i]))
    print(f"   {lam:>8.0f}{float(ztt_demo[i]):>11.6f}{float(delta_demo[i]):>+10.4f}"
          f"{rrs:>11.6f}{float(C.rrs_to_Rrs(rrs)):>11.6f}")

Two things to notice in those numbers, both of which the rest of the notebook turns
on. O25's coefficients barely move with solar zenith — $G_{w0}$ falls only from 0.0587
to 0.0525 between 0° and 60° — yet that small dependence is enough to hold its error
flat across the three angles where the backbone's grows by a factor two (§4). And the
hybrid's $\delta$ is a *few percent*: the physics sets the answer and the learned half
adjusts it, which is what makes the correction bounded rather than free.

## 2. The benchmark changed, and with it the claim

M3's table compared the hybrid against standard Gordon (1988) and the ZTT backbone.
M4 adds **O25**: `Rrs = (Gw0 + Gw1·ω_bw)·ω_bw + (Gp0 + Gp1·ω_bp)·ω_bp`, a bivariate
quadratic in the water- and particle-split pseudo-albedos, with four coefficients per
geometry.

Two things make it a fair fight and one makes it a hard one. Fair: its
water/particle split is exactly the split this project already keeps explicit, so
nothing has to be approximated to run it; and its coefficients are fitted on **our
training split** with **our own metric** as the objective, so it is being given its
best shot. Hard: that is twelve fitted numbers against the hybrid's 417.

The coefficients are *not* the published ones — the paper prints only plots — so
everything below says **"O25 form, refit on L23"**.

In [ ]:
batch_full = None
try:
    batch_full = L.load_batch()
except Exception as exc:
    print(f"L23 not mounted ($OS_COLOR) — falling back to the cached fixture: {exc}")
batch = batch_full if batch_full is not None else L.load_batch(
    reader=L.npz_reader(REPO / "robust/tests/files/l23_small.npz"))

splits = L.make_splits(batch)
wave, zen = np.asarray(batch.wave), batch.zenith
truth = C.Rrs_to_rrs(batch.Rrs)
args = (batch.iops, batch.phase_params, batch.geometry, batch.wave)
rrs_ztt = Z.rrs_ZTT(*args)
print(f"{batch.n_sample} samples x {batch.n_wave} lambda"
      f"   ({'full L23' if batch_full is not None else 'committed fixture'})")

mlp, _ = E.fit_l23(batch, splits, rrs_ztt=rrs_ztt)
lin, _ = E.fit_l23(batch, splits, config=E.LINEAR_CONFIG, rrs_ztt=rrs_ztt)
models = {
    "standard Gordon": B.rrs_gordon(*args),
    "ZTT backbone": rrs_ztt,
    "O25 form, refit on L23": B.rrs_o25(*args),
    "hybrid, linear": rrs_ztt * (1.0 + lin.relative_delta(*args)),
    "hybrid, MLP": rrs_ztt * (1.0 + mlp.relative_delta(*args)),
}

masks = {"train": splits.scene_train, "held-out scenes": splits.scene_test,
         "held-out @60": splits.scene_test & (zen == 60)}
table = V.score_models(models, truth, masks)
print(f"\n{'rRMS [%]':<26}" + "".join(f"{k:>17}" for k in masks))
print("-" * (26 + 17 * len(masks)))
for name, row in table.items():
    print(f"{name:<26}" + "".join(f"{row[k]:>17.2f}" for k in masks))

held = table["hybrid, MLP"]["held-out scenes"]
print(f"\nthe hybrid's margin over standard Gordon: "
      f"{table['standard Gordon']['held-out scenes'] / held:5.1f}x")
print(f"                          ... over O25:    "
      f"{table['O25 form, refit on L23']['held-out scenes'] / held:5.1f}x")

Twelve fitted numbers land within a factor of 2.3 of a 417-parameter network, and
O25's train and held-out figures are identical, so there is nothing to write off as
overfitting. **The 24× headline was a statement about Gordon, not about the hybrid.**

That is not a reason to be gloomy about the hybrid — it is the difference between a
claim that survives contact with a reviewer and one that does not.

In [ ]:
fig, ax = plt.subplots(figsize=(8.6, 5.0))
styles = {
    "standard Gordon": (C_C, 1.8, "-"), "ZTT backbone": (C_B, 1.8, "-"),
    "O25 form, refit on L23": (C_A, 2.2, "-"), "hybrid, linear": (MUTED, 1.6, "--"),
    "hybrid, MLP": (INK, 2.2, "-"),
}
test = splits.scene_test
for name, (colour, width, dash) in styles.items():
    ladder = np.asarray(V.rrms_per_wavelength(truth[test], models[name][test]))
    ax.plot(wave, ladder, color=colour, lw=width, ls=dash, label=name)
ax.set_yscale("log")
ax.set_xlim(wave.min(), wave.max())
ax.set_xlabel("wavelength [nm]")
ax.set_ylabel("rRMS [%]   (log)")
fig.legend(loc="lower center", ncols=3, fontsize=9.5, bbox_to_anchor=(0.5, -0.01))
fig.suptitle("Held-out scenes, per wavelength: the gap that matters is to the blue "
             "line,\nnot to the sky one", fontsize=12, x=0.01, ha="left")
fig.tight_layout(rect=(0, 0.10, 1, 0.93))
plt.show()

Gordon (sky) is the only model that degrades badly toward the red, which is what the
relative metric exists to expose. ZTT (vermillion) is flat but mediocre, and worst in
the 550 nm hump [notebook 3](rt_elastic_coding_3.ipynb) traced to its geometry term.
O25 (blue) sits a decade below both. The MLP hybrid (black) is below O25 at nearly
every wavelength — but by a factor, not a decade, and the two curves come within ~2×
of each other in the blue.

## 3. Was the comparison fair? The fitting objective is worth 4×

O25's coefficients have to come from somewhere, and the choice of objective decides
how strong the rival is. The paper fits **unweighted** least squares in `Rrs`. This
project scores everything with a **relatively weighted** rRMS. Fitting a rival with an
objective that does not match the metric it will be judged by is the easiest way to
win an unfair comparison, so it is worth measuring rather than assuming.

In [ ]:
fit_weighted = B.fit_o25(batch.iops, batch.Rrs, batch.geometry, train=splits.scene_train)
fit_paper = B.fit_o25(batch.iops, batch.Rrs, batch.geometry,
                      train=splits.scene_train, weighted=False)
o25_weighted = B.rrs_o25(*args, coeffs=fit_weighted)
o25_paper = B.rrs_o25(*args, coeffs=fit_paper)

print("O25 on held-out scenes, by fitting objective:")
for label, pred in (("relatively weighted (ours)", o25_weighted),
                    ("unweighted (the paper's)", o25_paper)):
    print(f"   {label:28s} {float(V.rrms(truth[test], pred[test])):5.2f}%")
print(f"\n   the hybrid, for scale:       "
      f"{table['hybrid, MLP']['held-out scenes']:5.2f}%")
print("\nso adopting the paper's objective would have made our own hybrid look")
print(f"{float(V.rrms(truth[test], o25_paper[test])) / float(V.rrms(truth[test], o25_weighted[test])):.1f}x"
      " better than a fair comparison allows.")

In [ ]:
fig, ax = plt.subplots(figsize=(8.4, 4.8))
for label, pred, colour, dash in (
    ("O25, unweighted fit (the paper's objective)", o25_paper, C_B, "--"),
    ("O25, relatively weighted fit (ours)", o25_weighted, C_A, "-"),
    ("hybrid, MLP", models["hybrid, MLP"], INK, "-"),
):
    ax.plot(wave, np.asarray(V.rrms_per_wavelength(truth[test], pred[test])),
            color=colour, lw=2.0, ls=dash, label=label)
ax.set_yscale("log")
ax.set_xlim(wave.min(), wave.max())
ax.set_xlabel("wavelength [nm]")
ax.set_ylabel("rRMS [%]   (log)")
ax.legend(loc="upper left", fontsize=9.5)
fig.suptitle("An unweighted fit abandons the dark red — which would have flattered\n"
             "our own model by 4x", fontsize=12, x=0.01, ha="left")
fig.tight_layout(rect=(0, 0, 1, 0.90))
plt.show()

The unweighted curve is *better in the blue* and far worse everywhere else: least
squares in `Rrs` spends its freedom where `Rrs` is large and abandons the dark red,
which is exactly the failure the BING lesson warns about and exactly what a relative
metric punishes. Both curves are honest fits of the same model — they differ only in
what they were asked to minimise.

So the fair fit is the default here, and the paper's sits behind `weighted=False`.
Note what this means for the headline: **O25's 0.69% is its best case**, obtained on
our training split with our own metric as the objective. That fact has to travel with
the number.

## 4. The breakdowns the protocol requires

In [ ]:
print("rRMS on held-out scenes, per solar zenith:")
print(f"{'':<26}" + "".join(f"{z:>10.0f}°" for z in (0, 30, 60)))
for name, pred in models.items():
    by_zen = V.group_rrms(truth[test], pred[test], zen[test])
    print(f"{name:<26}" + "".join(f"{by_zen[z]:>11.2f}" for z in (0.0, 30.0, 60.0)))

labels, edges = V.bp_bin_labels(batch.phase_params.B_p)
print(f"\nrRMS on held-out scenes, per B_p bin (equal counts):")
print(f"{'':<26}" + "".join(f"{f'{edges[i]:.4f}-{edges[i+1]:.4f}':>16}" for i in range(4)))
for name, pred in models.items():
    by_bin = V.group_rrms(truth[test], pred[test], labels[test])
    print(f"{name:<26}" + "".join(f"{by_bin[float(i)]:>16.2f}" for i in range(4)))
print(f"\nthose bins span a factor {edges[-1]/edges[0]:.2f} in B_p, against the "
      "design's ~7x nominal band.")

The zenith cut is where the analytic backbone's weakness shows: ZTT degrades from
4.3% at 0° to 8.1% at 60°, the geometry error [notebook 3](rt_elastic_coding_3.ipynb)
diagnosed. O25 and the hybrid are flat, because both have a per-geometry handle — O25
a coefficient row, the hybrid a `cos θ_s` feature.

The `B_p` cut is required by design §6 and is close to uninformative, which is worth
saying rather than presenting as a passed check: **L23 spans a factor 1.7 in `B_p`
against the design's ~7×**, so these four bins are four samples from a narrow slice.
Flat accuracy across them is *not* evidence of phase-function generalisation. That
test needs M5's HydroLight runs.

## 5. Out of distribution: the unseen zenith

Everything above is interpolation — every model saw all three solar zeniths. The
harder question is the CQ6 geometry split: train on 0°/30°, score the unseen 60°. For
anything trained, one number would be a claim about a seed, so this is run five times.

In [ ]:
SEEDS = (23, 1, 7, 101, 2024)
tr, te = splits.zenith_train, splits.zenith_test

o25_z = B.rrs_o25(*args, coeffs=B.fit_o25(
    batch.iops, batch.Rrs, batch.geometry, train=tr, zeniths=(0.0, 30.0)))
rows = [
    ("standard Gordon", [float(V.rrms(truth[te], B.rrs_gordon(*args)[te]))]),
    ("ZTT backbone", [float(V.rrms(truth[te], rrs_ztt[te]))]),
    ("O25 form, refit on 0/30", [float(V.rrms(truth[te], o25_z[te]))]),
]
for label, hidden in (("hybrid, linear", ()), ("hybrid, MLP", (16, 16))):
    scores = []
    for seed in SEEDS:
        em, _ = E.fit(*args, truth, train=tr, rrs_ztt=rrs_ztt,
                      config=E.EmulatorConfig(hidden=hidden, seed=seed))
        scores.append(float(V.rrms(truth[te], (rrs_ztt * (1 + em.relative_delta(*args)))[te])))
    rows.append((label, scores))

print(f"rRMS at the unseen 60°, over seeds {SEEDS}:")
for name, scores in rows:
    spread = "" if len(scores) == 1 else (
        f"   [{min(scores):5.2f} – {max(scores):5.2f}]" if max(scores) > min(scores)
        else "   [identical for every seed]")
    print(f"   {name:<26}{np.median(scores):5.2f}%{spread}")

In [ ]:
fig, ax = plt.subplots(figsize=(8.6, 4.4))
positions = np.arange(len(rows))
for y, (name, scores) in zip(positions, rows):
    colour = INK if "MLP" in name else (C_A if "O25" in name else INK_MUTED)
    lo, hi, median = min(scores), max(scores), float(np.median(scores))
    if hi > lo:
        ax.plot([lo, hi], [y, y], color=colour, lw=6, alpha=0.30, solid_capstyle="butt")
        label = f"{lo:.2f}–{hi:.2f} over seeds"
    else:
        label = f"{median:.2f}" + ("  (same for every seed)" if len(scores) > 1 else "")
    ax.plot([median], [y], "o", color=colour, ms=8)
    ax.annotate(label, xy=(hi + 0.35, y), va="center", fontsize=9, color=colour)

ax.set_yticks(positions, [name for name, _ in rows], fontsize=9.5)
ax.invert_yaxis()
ax.set_xlim(0, max(max(s) for _, s in rows) * 1.5)
ax.set_xlabel("rRMS at the unseen 60° [%]   (dot = median over seeds)")
fig.suptitle("Trained on 0°/30°, scored at the unseen 60°: the refit O25 wins, and\n"
             "only the MLP's answer depends on its seed",
             fontsize=12, x=0.01, ha="left")
fig.tight_layout(rect=(0, 0, 1, 0.88))
plt.show()

This is the figure the milestone turns on. The MLP hybrid's error at an unseen
geometry spans a factor 2.6 across five identical configurations that differ only in
initialisation — and the band straddles Gordon's 9.01%, so *whether the prototype
passes the plan's original gate is decided by a random seed*. The refit O25 sits at
4.63%, deterministically, below the hybrid's best seed.

So M4's gate is written on the **scene** split (where the hybrid beats O25, 0.30%
against 0.69%) and this split is **reported, not gated** — JXP's call, and the honest
one.

One consequence worth spelling out, because it is not obvious. The out-of-domain
fallback added for exactly this situation — `on_out_of_domain="ztt"`, which degrades
the hybrid to the backbone outside the sanctioned range — **does not fire here**. The
sanctioned envelope is 0–60°, so 60° is *inside* it by decision, even though it is
outside what this particular emulator was trained on.

In [ ]:
em30, _ = E.fit(*args, truth, train=tr, rrs_ztt=rrs_ztt,
                config=E.EmulatorConfig(seed=2024))
plain = H.rrs_forward(*args, "hybrid", emulator=em30, check_domain=False)
guarded = H.rrs_forward(*args, "hybrid", emulator=em30, check_domain=False,
                        on_out_of_domain="ztt")
n_flagged = int(np.asarray(em30.out_of_domain_mask(*args)).sum())
print(f"samples the fallback would rescue at 0-60 deg: {n_flagged} of {batch.n_sample}")
print(f"unseen-60 rRMS, policy off: {float(V.rrms(truth[te], plain[te])):.2f}%")
print(f"unseen-60 rRMS, policy on : {float(V.rrms(truth[te], guarded[te])):.2f}%")
print(f"\nand at 75 deg, which IS beyond the envelope ({E.SUPPORTED_THETA_S[1]:.0f} deg):")
low_sun = (batch.iops, batch.phase_params,
           Geometry.nadir(jnp.full_like(batch.geometry.theta_s, 75.0)), batch.wave)
print(f"   flagged: {int(np.asarray(em30.out_of_domain_mask(*low_sun)).sum())} "
      f"of {batch.n_sample}, and the correction is then dropped entirely")

The fallback is still the right thing beyond 60° — it is what makes "we will not use
the emulator at larger angles" a property of the code rather than a promise in a
document. It simply cannot rescue this half of the gate.

## 6. Fast, and differentiable

The other two axes of design §6. Both are gates rather than comparisons: a model that
is accurate but slow, or accurate but not differentiable, fails the project's purpose.

In [ ]:
timings = {}
for name, fn in (
    ("standard Gordon", lambda i, p, g, w: B.rrs_gordon(i, p, g, w)),
    ("ZTT backbone", lambda i, p, g, w: Z.rrs_ZTT(i, p, g, w)),
    ("O25 form", lambda i, p, g, w: B.rrs_o25(i, p, g, w)),
    ("hybrid, MLP", lambda i, p, g, w: H.rrs_forward(
        i, p, g, w, "hybrid", emulator=mlp, check_domain=False)),
):
    timings[name] = V.throughput(fn, *args)
reference = timings["ZTT backbone"][0]
print(f"{'jitted, full batch':<20}{'ms/call':>10}{'M sample-λ/s':>15}{'x ZTT':>9}")
for name, (seconds, rate) in timings.items():
    print(f"{name:<20}{seconds*1e3:>10.2f}{rate/1e6:>15.0f}{seconds/reference:>9.2f}")
print("\n(both columns wander between runs -- the hybrid has measured 4.5-6.0x ZTT\n across four runs -- so the ORDERING is what to rely on, not either number)")

In [ ]:
jax.config.update("jax_enable_x64", True)
sel = np.where(zen == 30)[0][:3]
f64 = lambda x: jnp.asarray(np.asarray(x)[sel], dtype=jnp.float64)
iops64 = IOPs(a=f64(batch.iops.a), bb_w=f64(batch.iops.bb_w), bb_p=f64(batch.iops.bb_p))
phase64 = PhaseParams(B_p=f64(batch.phase_params.B_p))
wave64 = jnp.asarray(wave, dtype=jnp.float64)

for label, angle in (("at 45° (between O25's table nodes)", 45.0),
                     ("at 30° (ON a node)", 30.0)):
    geom = Geometry.nadir(jnp.full((3,), angle, dtype=jnp.float64))
    print(f"{label}:")
    for name, fn in (
        ("ZTT backbone", lambda i, p, g, w: Z.rrs_ZTT(i, p, g, w)),
        ("O25 form", lambda i, p, g, w: B.rrs_o25(i, p, g, w)),
        ("hybrid, MLP", lambda i, p, g, w: H.rrs_forward(
            i, p, g, w, "hybrid", emulator=mlp, check_domain=False)),
    ):
        rep = V.gradient_report(fn, iops64, phase64, geom, wave64)
        print(f"   {name:<14}" + "  ".join(f"{k} {rep[k]:.0e}" for k in V.FD_STEPS))
jax.config.update("jax_enable_x64", False)

Speed: the hybrid costs **4.5–6× the backbone** — the spread is across repeated runs
on this machine, not across configurations — and still puts the full 9960 × 81 batch
through in under 20 ms. The learned half dominates that cost — it evaluates a 417-parameter
network at each of 806,760 points — and the advantage over calling an RT solver
survives with room to spare.

Gradients: every model, every variable, agrees with central differences to ~1e-9 at
45°, far inside the 1e-6 gate.

**And the second block shows a trap worth knowing.** At 30° O25's `theta_s` gradient
disagrees by ~70%. Nothing is broken: its coefficient lookup is piecewise linear, so
the tabulated angles are *kinks*, where `jax.grad` takes one one-sided slope and a
central difference averages both. L23's three angles **are** the nodes, so a
finite-difference check run on L23 geometry lands on one every time. `B_p` reads 0 for
O25 for a different reason — it genuinely ignores the phase function, so both
derivatives are exactly zero, which is agreement rather than failure. An earlier
version of the report divided by the finite difference and called that infinitely
wrong.

## 7. What the prototype may and may not say

**May say.** On held-out water bodies, the hybrid reaches **0.30% rRMS**, against
0.69% for a strong semi-analytical benchmark refit on the same data, 5.93% for the
analytic backbone alone and 7.21% for standard Gordon. It is uniform across the
spectrum and across the three solar zeniths, it is differentiable to ~1e-9 in every
input, and it costs ~6× the backbone. The gain is generalisation, not fit: held-out
equals train to two decimals.

**May not say.** That it beats the state of the art by 24× — that is the margin over
a 1988 model. That it extrapolates in geometry: at an unseen solar zenith its error
depends on the random seed (4.7–12.2%) and the refit O25 beats it deterministically
at 4.63%. That it generalises across phase functions: L23 offers a 1.7× slice of
`B_p` and a single fixed Fournier-Forand shape, so §4's flat bins are not evidence.
And O25's own 0.69% is its best case, fitted on our training split with our metric.

**Still open**, carried into M5: the phase-function axis (M5's HydroLight runs are
what would test it), the geometry axis beyond a single unseen angle, and the
published O25/PR05 coefficients — every O25 number here is a refit, labelled as such.